# Notebook 02 - Modelagem: Resolucao de Chamados em 7 Dias

## Resumo Executivo

Este notebook apresenta a modelagem preditiva para determinar se um chamado do sistema 1746
sera resolvido em ate 7 dias. Foram treinados e comparados 5 modelos:

- **Regressao Logistica** (baseline)
- **Random Forest** (200 arvores)
- **XGBoost** (200 arvores, default + otimizado com Optuna)
- **LightGBM** (200 arvores)

**Melhor modelo**: XGBoost (tuned) com F1=0.8941 e AUC-ROC=0.8628.

A analise de interpretabilidade via SHAP revelou que as features mais importantes sao
relacionadas a tipo de servico, localizacao e padroes temporais.

---

## Q6: Modelo Baseline - Regressao Logistica

### Justificativa da Metrica

No contexto de politicas publicas para cidadaos vulneraveis:
- **Recall** e critico: nao identificar um chamado que NAO sera resolvido em 7 dias significa
  um cidadao vulneravel sem atendimento.
- **Precisao** importa para alocacao eficiente de recursos limitados.
- **F1-score** e a metrica primaria (equilibra precisao e recall).
- **AUC-ROC** e a metrica secundaria para comparacao geral.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

# Carregar dados
X_train = pd.read_parquet('data/features/X_train.parquet')
X_test = pd.read_parquet('data/features/X_test.parquet')
y_train = pd.read_parquet('data/features/y_train.parquet').squeeze()
y_test = pd.read_parquet('data/features/y_test.parquet').squeeze()

print(f"Treino: {X_train.shape[0]} amostras, {X_train.shape[1]} features")
print(f"Teste:  {X_test.shape[0]} amostras")
print(f"Taxa positiva (treino): {y_train.mean():.3f}")
print(f"Taxa positiva (teste):  {y_test.mean():.3f}")

In [ ]:
from src.models.train_baseline import train_logistic_baseline, evaluate_model

model_lr = train_logistic_baseline(X_train, y_train)
metrics_lr = evaluate_model(model_lr, X_test, y_test)

print("=== Regressao Logistica (Baseline) ===")
for k in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'auc_pr']:
    print(f"  {k:12s}: {metrics_lr[k]:.4f}")

### Curva ROC e Matriz de Confusao do Baseline

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/figures/q6_roc_baseline.png'))

In [ ]:
display(Image(filename='results/figures/q6_confusion_matrix.png'))

## Q7: Modelos Avancados e Otimizacao de Hiperparametros

Treinamos Random Forest, XGBoost e LightGBM com configuracoes padroes (200 arvores cada).
Em seguida, otimizamos o XGBoost com Optuna (50 trials, 5-fold stratified CV, maximizando AUC-ROC).

In [ ]:
# Tabela de comparacao completa
import pandas as pd
data = {'Modelo': ['Logistic Regression', 'Random Forest', 'XGBoost (default)', 'LightGBM', 'XGBoost (tuned)'], 'Accuracy': ['0.8259', '0.8268', '0.8137', '0.8237', '0.8283'], 'Precision': ['0.8607', '0.8651', '0.8613', '0.8669', '0.8645'], 'Recall': ['0.9277', '0.9226', '0.9084', '0.9153', '0.9259'], 'F1': ['0.8930', '0.8929', '0.8842', '0.8905', '0.8941'], 'AUC-ROC': ['0.8480', '0.8493', '0.8508', '0.8609', '0.8628'], 'AUC-PR': ['0.9462', '0.9535', '0.9566', '0.9597', '0.9602']}
comp_df = pd.DataFrame(data)
comp_df.style.highlight_max(subset=['F1', 'AUC-ROC', 'AUC-PR'], color='lightgreen')

### Melhores Hiperparametros (Optuna - XGBoost)

```json
{
  "max_depth": 9,
  "learning_rate": 0.018596846637774906,
  "n_estimators": 288,
  "subsample": 0.8739532290914787,
  "colsample_bytree": 0.8819912795715482,
  "min_child_weight": 2,
  "reg_alpha": 0.023798833430266154,
  "reg_lambda": 4.191562880920944e-06
}
```

### Resultado: Melhor Modelo = **XGBoost (tuned)**

| Metrica   | Valor  |
|-----------|--------|
| F1        | 0.8941 |
| AUC-ROC   | 0.8628 |
| AUC-PR    | 0.9602 |
| Precision | 0.8645 |
| Recall    | 0.9259 |

### Curvas ROC e Precision-Recall (todos os modelos)

In [ ]:
display(Image(filename='results/figures/q7_roc_curves.png'))

In [ ]:
display(Image(filename='results/figures/q7_pr_curves.png'))

In [ ]:
display(Image(filename='results/figures/q7_confusion_best.png'))

In [ ]:
display(Image(filename='results/figures/q7_model_comparison.png'))

## Q8: Interpretabilidade e Analise de Erros

### Analise SHAP (TreeExplainer)

Utilizamos o SHAP (SHapley Additive exPlanations) para interpretar as predicoes do
melhor modelo (XGBoost (tuned)). O SHAP atribui a cada feature uma contribuicao marginal
para cada predicao individual.

In [ ]:
display(Image(filename='results/figures/q8_shap_summary_beeswarm.png'))

### Top 10 Features por Importancia SHAP

In [ ]:
display(Image(filename='results/figures/q8_shap_bar_top10.png'))

### Graficos de Dependencia SHAP

As 3 features mais importantes segundo SHAP: **subtipo_encoded, orgao_encoded, hist_resolution_rate_bairro**

In [ ]:
display(Image(filename='results/figures/q8_shap_dependence_subtipo_encoded.png'))

In [ ]:
display(Image(filename='results/figures/q8_shap_dependence_orgao_encoded.png'))

In [ ]:
display(Image(filename='results/figures/q8_shap_dependence_hist_resolution_rate_bairro.png'))

### Importancia Nativa do Modelo

In [ ]:
display(Image(filename='results/figures/q8_feature_importance.png'))

### Analise de Erros

| Tipo | Quantidade | Percentual |
|------|-----------|------------|
| Falsos Positivos (FP) | 2950 | 11.4% |
| Falsos Negativos (FN) | 1506 | 5.8% |

- **FP**: modelo previu resolucao em 7 dias, mas o chamado NAO foi resolvido.
- **FN**: modelo previu NAO resolucao, mas o chamado FOI resolvido.

In [ ]:
display(Image(filename='results/figures/q8_error_analysis.png'))

### Insights para Politicas Publicas

1. **Padroes temporais** (hora, dia, mes) influenciam fortemente a probabilidade de resolucao,
   sugerindo que o momento de abertura do chamado afeta a capacidade de resposta municipal.

2. **Features territoriais** (bairro, subprefeitura) mostram variacao significativa,
   indicando disparidades geograficas na prestacao de servicos.

3. **Tipo de servico** (tipo, subtipo, orgao) captura diferencas de capacidade institucional
   entre as agencias municipais.

4. **Taxas historicas de resolucao** por bairro fornecem forte sinal preditivo,
   refletindo o desempenho institucional acumulado.

5. **Condicoes meteorologicas** tem influencia moderada, com eventos de chuva extrema
   correlacionados com atrasos (sobrecarga de infraestrutura).

**Recomendacao**: O sistema de priorizacao (Q9-Q10) deve priorizar a reducao de FN,
pois nao identificar um chamado atrasado significa um cidadao vulneravel sem atendimento.

---

*Notebook gerado automaticamente pelo Agent 4 (Model Builder).*